In [1]:
#climate/weather

!pip install pandas
!pip install polars
!pip install dask
!pip install pyarrow
!pip install duckdb

In [2]:
import dask.dataframe as dd
import dask.dataframe as dd
import pyarrow as pa
import polars as ps
import numpy as np
import pandas as pd
import glob
import os
import duckdb

In [3]:
FILE = '/projects/illinois/ovcri/ncsa/cs2018/idph-ch-datasets/climate/ERA5/era5_all_years.parquet'

In [4]:
# Connect to DuckDB (runs in memory, no setup needed)
con = duckdb.connect()

In [5]:
print("=== COLUMN NAMES & TYPES ===")
print(con.execute(f"DESCRIBE SELECT * FROM '{FILE}'").df())

print("\n=== TOTAL ROWS ===")
count = con.execute(f"SELECT COUNT(*) FROM '{FILE}'").fetchone()[0]
print(f"  {count:,} rows")

print("\n=== FIRST 5 ROWS ===")
print(con.execute(f"SELECT * FROM '{FILE}' LIMIT 5").df())

=== COLUMN NAMES & TYPES ===
            column_name column_type null   key default extra
0          climate_date        DATE  YES  None    None  None
1                region     VARCHAR  YES  None    None  None
2            state_fips      BIGINT  YES  None    None  None
3                  fips      BIGINT  YES  None    None  None
4         temperature_c      DOUBLE  YES  None    None  None
5            dewpoint_c      DOUBLE  YES  None    None  None
6          heat_index_c      DOUBLE  YES  None    None  None
7   total_precipitation      DOUBLE  YES  None    None  None
8     relative_humidity      DOUBLE  YES  None    None  None
9              sfc_wind      DOUBLE  YES  None    None  None
10     surface_pressure      DOUBLE  YES  None    None  None

=== TOTAL ROWS ===
  335,274 rows

=== FIRST 5 ROWS ===
  climate_date     region  state_fips   fips  temperature_c  dewpoint_c  \
0   2017-01-01      adams          17  17001        -1.5239     -6.4565   
1   2017-01-01  alexander       

In [6]:
print("=== NULL CHECK ===")
nulls = con.execute(f"""
SELECT 
    COUNT(*) as total_rows,
    {', '.join([f"COUNT(*) - COUNT({col}) AS {col}_nulls" for col in con.execute(f"DESCRIBE SELECT * FROM '{FILE}'").df()['column_name']])}
FROM '{FILE}'
""").df()

print(nulls)

=== NULL CHECK ===
   total_rows  climate_date_nulls  region_nulls  state_fips_nulls  fips_nulls  \
0      335274               37230             0                 0           0   

   temperature_c_nulls  dewpoint_c_nulls  heat_index_c_nulls  \
0                37230                 0                   0   

   total_precipitation_nulls  relative_humidity_nulls  sfc_wind_nulls  \
0                          0                        0               0   

   surface_pressure_nulls  
0                       0  


In [7]:
print("=== NUMERIC SUMMARY ===")
print(con.execute(f"""
SELECT * FROM '{FILE}'
""").df().describe())

=== NUMERIC SUMMARY ===
                     climate_date  state_fips           fips  temperature_c  \
count                      298044    335274.0  335274.000000  298044.000000   
mean   2020-12-31 11:59:59.999999        17.0   17102.000000      12.878773   
min           2017-01-01 00:00:00        17.0   17001.000000     -29.809600   
25%           2019-01-01 00:00:00        17.0   17051.000000       4.413025   
50%           2020-12-31 12:00:00        17.0   17102.000000      13.596100   
75%           2023-01-01 00:00:00        17.0   17153.000000      22.351300   
max           2024-12-31 00:00:00        17.0   17203.000000      32.295100   
std                           NaN         0.0      58.886985      10.547017   

          dewpoint_c   heat_index_c  total_precipitation  relative_humidity  \
count  335274.000000  335274.000000         3.352740e+05      335274.000000   
mean        6.756962      17.657357         4.172184e-04           8.793079   
min       -34.143500    -81

In [8]:
cols = con.execute(f"DESCRIBE SELECT * FROM '{FILE}'").df()['column_name']

for col in cols:
    print(f"\n=== {col} UNIQUE VALUES ===")
    print(con.execute(f"""
    SELECT {col}, COUNT(*) as count
    FROM '{FILE}'
    GROUP BY {col}
    ORDER BY count DESC
    LIMIT 5
    """).df())


=== climate_date UNIQUE VALUES ===
  climate_date  count
0          NaT  37230
1   2017-01-01    102
2   2017-01-02    102
3   2017-01-03    102
4   2017-01-04    102

=== region UNIQUE VALUES ===
       region  count
0   effingham   3287
1    kankakee   3287
2  livingston   3287
3      monroe   3287
4      bureau   3287

=== state_fips UNIQUE VALUES ===
   state_fips   count
0          17  335274

=== fips UNIQUE VALUES ===
    fips  count
0  17001   3287
1  17003   3287
2  17005   3287
3  17007   3287
4  17009   3287

=== temperature_c UNIQUE VALUES ===
   temperature_c  count
0            NaN  37230
1        24.1460      8
2        22.3607      8
3        23.4821      8
4        25.7845      8

=== dewpoint_c UNIQUE VALUES ===
   dewpoint_c  count
0     19.9066      8
1     -1.6707      8
2      0.6617      8
3     17.3831      8
4     -2.9677      8

=== heat_index_c UNIQUE VALUES ===
   heat_index_c  count
0       19.7592      8
1        8.4546      8
2        4.3889      7
3    

In [9]:
desc = con.execute(f"""
DESCRIBE SELECT * FROM '{FILE}'
""").df()

cols = desc['column_name'].tolist()

for col in cols:
    print(f"\n=== Checking {col} ===")
    
    print(con.execute(f"""
    SELECT {col}, COUNT(*) as count
    FROM '{FILE}'
    WHERE {col} IS NULL
    GROUP BY {col}
    LIMIT 5
    """).df())


=== Checking climate_date ===
  climate_date  count
0          NaT  37230

=== Checking region ===
Empty DataFrame
Columns: [region, count]
Index: []

=== Checking state_fips ===
Empty DataFrame
Columns: [state_fips, count]
Index: []

=== Checking fips ===
Empty DataFrame
Columns: [fips, count]
Index: []

=== Checking temperature_c ===
   temperature_c  count
0            NaN  37230

=== Checking dewpoint_c ===
Empty DataFrame
Columns: [dewpoint_c, count]
Index: []

=== Checking heat_index_c ===
Empty DataFrame
Columns: [heat_index_c, count]
Index: []

=== Checking total_precipitation ===
Empty DataFrame
Columns: [total_precipitation, count]
Index: []

=== Checking relative_humidity ===
Empty DataFrame
Columns: [relative_humidity, count]
Index: []

=== Checking sfc_wind ===
Empty DataFrame
Columns: [sfc_wind, count]
Index: []

=== Checking surface_pressure ===
Empty DataFrame
Columns: [surface_pressure, count]
Index: []


In [10]:
champaign_df = con.execute(f"""
SELECT *
FROM '{FILE}'
WHERE LOWER(Region) = 'champaign'
""").df()

print(champaign_df.head())
print(f"Total rows: {len(champaign_df)}")

  climate_date     region  state_fips   fips  temperature_c  dewpoint_c  \
0   2017-01-01  champaign          17  17019        -1.7606     -6.3316   
1   2017-01-02  champaign          17  17019         3.6459      1.7155   
2   2017-01-03  champaign          17  17019         6.2778      5.5792   
3   2017-01-04  champaign          17  17019        -6.2646    -11.3206   
4   2017-01-05  champaign          17  17019        -9.8939    -15.4292   

   heat_index_c  total_precipitation  relative_humidity  sfc_wind  \
0       -2.1385         2.579552e-09           1.433367  2.041462   
1        3.0726         1.662471e-04           1.151502  3.116697   
2        5.7020         5.985216e-04           1.049679  3.156478   
3       -6.8879         2.449995e-06           1.511574  6.729831   
4      -10.7118         5.658075e-05           1.576343  3.828583   

   surface_pressure  
0         99173.520  
1         99136.586  
2         98356.450  
3         99194.460  
4         99276.690  
To

In [11]:
ERA5_FILE = '/projects/illinois/ovcri/ncsa/cs2018/idph-ch-datasets/climate/ERA5/era5_all_years.parquet'

con.execute(f"""
COPY (
    SELECT *
    FROM '{ERA5_FILE}'
    WHERE LOWER(REPLACE(TRIM(region), ' ', '')) IN (
        'champaign','clark','coles','cumberland','dewitt','douglas','edgar',
        'ford','iroquois','livingston','piatt','macon','mclean',
        'moultrie','shelby','vermilion'
    )
    AND YEAR(climate_date) BETWEEN 2017 AND 2025
) TO 'era5_filtered.parquet' (FORMAT PARQUET)
""")

In [12]:
con.execute("""
COPY (
    SELECT 
        LOWER(REPLACE(TRIM(region), ' ', '')) AS county,
        climate_date AS date,
        AVG(temperature_c) AS avg_temp,
        AVG(total_precipitation) AS avg_precip
    FROM 'era5_filtered.parquet'
    GROUP BY county, date
) TO 'era5_agg.parquet' (FORMAT PARQUET)
""")